# NovaGates Onboarding — Milestone: MongoDB Query Notebook
**Programme:** 2-Week Backend & AI Developer (Junior Edition)
**Milestone:** Day 3 MongoDB Queries, Aggregations, Text & Vector Search
**Database:** `portfolio_db`

## 1. Setup & Connection
Run this notebook with a Python kernel or execute queries via mongosh.

In [ ]:
# Optional Python driver check using pymongo
# pip install pymongo
import json
from pprint import pprint

print('Environment ready for MongoDB operations.')

## Part 1: CRUD Operations
### Query 1: Single & Bulk Insert (`insertOne`, `insertMany`)

In [ ]:
/* mongosh syntax */
// 1.1 Insert a single skill
db.skills.insertOne({
  name: 'Celery',
  category: 'Backend',
  proficiency: 82,
  years_experience: 2,
  tags: ['async', 'queue', 'distributed', 'python']
});

// 1.2 Insert multiple skills in one round trip
db.skills.insertMany([
  {
    name: 'PostgreSQL',
    category: 'Database',
    proficiency: 78,
    years_experience: 2,
    tags: ['relational', 'sql', 'acid']
  },
  {
    name: 'Tavily API',
    category: 'AI / ML',
    proficiency: 85,
    years_experience: 1,
    tags: ['search', 'ai', 'rag', 'agents']
  }
]);

### Query 2: Finding Documents with Projection
Fetch completed projects without loading vector embeddings.

In [ ]:
db.projects.find(
  { status: 'completed' },
  { title: 1, slug: 1, stars: 1, status: 1, _id: 1 }
);

### Query 3: Atomic Updates (`updateOne`, `updateMany`)

In [ ]:
// Increment stars atomically and add timestamp
db.projects.updateOne(
  { slug: 'smart-job-matcher' },
  {
    $inc: { stars: 1 },
    $set: { last_reviewed_at: new Date() }
  }
);

// Add tag without duplication using $addToSet
db.skills.updateMany(
  { category: 'AI / ML' },
  { $addToSet: { tags: 'generative-ai' } }
);

### Query 4: Deletion (`deleteMany`)

In [ ]:
db.skills.deleteMany({
  category: 'Draft',
  proficiency: { $lt: 50 }
});

## Part 2: Filtering & Cursor Operations
### Query 5: Complex Filter ($and, $or, $in, $gt, $regex)

In [ ]:
db.skills.find({
  $and: [
    { proficiency: { $gte: 80 } },
    {
      $or: [
        { category: { $in: ['Backend', 'Database'] } },
        { tags: { $regex: /^ai/i } }
      ]
    },
    { years_experience: { $exists: true, $ne: null } }
  ]
});

### Query 6: Cursor Pagination (Sort, Skip, Limit)

In [ ]:
db.projects.find({ featured: true })
  .sort({ stars: -1, title: 1 })
  .skip(0)
  .limit(2);

## Part 3: Indexes & Explain Plan
### Query 7 & 8: Compound Indexing & Query Execution Stats

In [ ]:
// Create compound index for fast status and ranking queries
db.projects.createIndex({ status: 1, stars: -1 });

// Verify execution plan uses IXSCAN
db.projects.find({ status: 'completed' })
  .sort({ stars: -1 })
  .explain('executionStats');

## Part 4: Aggregation Pipelines (3+ Queries)
### Query 9: Skill Distribution ($unwind, $group, $project, $sort)

In [ ]:
db.skills.aggregate([
  { $unwind: '$tags' },
  {
    $group: {
      _id: '$category',
      total_skills: { $addToSet: '$name' },
      avg_proficiency: { $avg: '$proficiency' },
      all_tags: { $addToSet: '$tags' }
    }
  },
  {
    $project: {
      _id: 0,
      category: '$_id',
      skill_count: { $size: '$total_skills' },
      skills: '$total_skills',
      average_proficiency: { $round: ['$avg_proficiency', 1] },
      tags: '$all_tags'
    }
  },
  { $sort: { skill_count: -1 } }
]);

### Query 10: Relational Join with $lookup

In [ ]:
db.projects.aggregate([
  { $match: { featured: true } },
  {
    $lookup: {
      from: 'skills',
      localField: 'skill_ids',
      foreignField: '_id',
      as: 'skill_details'
    }
  },
  {
    $addFields: {
      skill_names: '$skill_details.name',
      primary_skill_count: { $size: '$skill_details' }
    }
  },
  {
    $project: {
      title: 1,
      slug: 1,
      status: 1,
      stars: 1,
      skill_names: 1,
      primary_skill_count: 1
    }
  },
  { $sort: { stars: -1 } }
]);

### Query 11: Work Experience Aggregation

In [ ]:
db.experiences.aggregate([
  { $match: { company: { $exists: true } } },
  {
    $project: {
      company: 1,
      role: 1,
      is_current: 1,
      tech_count: { $size: '$technologies' },
      technologies: 1,
      responsibilities_count: { $size: '$responsibilities' }
    }
  },
  { $sort: { is_current: -1, company: 1 } }
]);

## Part 5: Full-Text Search
### Query 12: Text Index with Weighted Scoring

In [ ]:
// Create text index
db.projects.createIndex(
  { title: 'text', summary: 'text', description: 'text' },
  { weights: { title: 10, summary: 5, description: 1 }, name: 'proj_text_idx' }
);

// Search using $text
db.projects.find(
  { $text: { $search: 'agent celery background' } },
  { score: { $meta: 'textScore' }, title: 1, summary: 1 }
).sort({ score: { $meta: 'textScore' } });

## Part 6: MongoDB Atlas Vector Search
### Query 13: Semantic Search with $vectorSearch

In [ ]:
db.projects.aggregate([
  {
    $vectorSearch: {
      index: 'vector_index',
      path: 'embedding',
      queryVector: [0.480, 0.190, -0.090, 0.620, 0.410, -0.080, 0.290, 0.490],
      numCandidates: 50,
      limit: 3,
      filter: {
        status: { $in: ['completed', 'in-progress'] }
      }
    }
  },
  {
    $project: {
      _id: 1,
      title: 1,
      summary: 1,
      status: 1,
      similarity_score: { $meta: 'vectorSearchScore' }
    }
  }
]);